# Importación

In [3]:
import os
import re
import pandas as pd

## Carpeta Local

In [4]:
folder = "cuestionarios"

## Carpeta en Drive (con Colab)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Obtener todos los xlsx de la carpeta compartida
folder = "/content/drive/MyDrive/Cuestionarios"

ModuleNotFoundError: No module named 'google'

## Proceso de Importación

In [7]:
files = os.listdir(folder)
excel_files = sorted([f for f in files if f.endswith('.xlsx')])

# Guardar todos los datos como dataframes, dentro de una lista de dataframes
questionnaires = [] #Aplica la limpieza a cada cuestionario
filenames = []      #

for file in excel_files:
  file_path = os.path.join(folder, file) #Busca los archivos en la carpeta
  file_df = pd.read_excel(file_path)     #Lee los datos de los archivos
  # strip() sobre los nombres de columnas
  file_df.columns = file_df.columns.str.strip()  #Definimos los encabezados
  questionnaires.append(file_df)                 #Crea un arreglo
  filenames.append(os.path.basename(file_path))  #

# Guía para saber qué número corresponde a qué archivo
for filename in filenames:
  print(f"{filenames.index(filename)} es {filename}")

0 es CaminoProfesionalMAC1.xlsx
1 es Finanzas Transparentes (Respuestas).xlsx
2 es Habilidades de MAC (respuestas).xlsx
3 es Impacto del Horario en el Desempeño y Bienestar de los Estudiantes de Minería de Datos (respuestas).xlsx
4 es Perfil Académico (respuestas).xlsx
5 es Rendimiento académico (respuestas).xlsx
6 es Uso de Tecnología y Redes Sociales.xlsx


# Clasificación

## Función de Clasificación

In [8]:
# Función que clasifica una respuesta dada un mapeo palabras -> categoría
def classify_answer(answer, mapping):

  # ¿Qué es esto y para qué está?
  """
  # Precompilamos las expresiones regulares para cada categoría
  compiled_patterns = {
    classification: [re.compile(pattern, re.IGNORECASE) for pattern in patterns]
    for classification, patterns in mapping.items()
  }
  """

  # Si la respuesta es nula
  if pd.isna(answer):
    return "nulo"

  for classification, patterns in mapping.items():
    for pattern in patterns:
      if re.search(pattern, answer, re.IGNORECASE):
        return classification
  # unclassified
  return "u"

## Mappings

(solían ser mappings universales, càd clasificaciones que aparecen en varias preguntas en varios cuestionarios; ahora son (casi) todos los mappings en un solo lugar)
(¿conviene dejarlo así o poner cada mapping después de su respectiva pregunta?)

In [9]:
# Mapeo para clasificación
# clasificación: [lista de palabras a buscar en la respuesta para clasificar]
# (palabras son en realidad expresiones regulares)

mapping_intensity = {
    "nada": ["nada", "no afecta", "ninguno", "no impacta", "ninguna"],
    "muy poco": ["poco", "no tanto", "deficiente", "puede mejorar", "sigo trabajando", "miedo"],
    "poco": ["algo", "masomenos", "medianamente", "regular", "me defiendo", "intermedio", "aveces bien", "a veces"],
    "moderado": ["suficiente", "bien", "preparad", "la mayoría de veces bien", "c.mod"],
    "mucho": ["mucho", "bastante", "bastante preparado"],
    "muy alto": ["super bien","demasiado", "considerablemente", "gran", "m.s", "alta", "completamente", "muy"],
    "desconocido": ["no s", "no lo sé", "ya veremos", "depende"]
}

mapping_yes_no = {
    "sí": ["sí", "si"],
    "no": ["no"]
}

mapping_gender = {
    "masculino": ["masculino", "hombre"],
    "femenino": ["femenino", "mujer"],
    "otro": ["no binario", ".+"]
}

mapping_soft_skills = {
    "comunicación": ["comunica", "explic", "expres", "ponencia", "habl","expo"],
    "trabajo en equipo": ["equipo", "cooper", "integr", " dem.s"],
    "liderazgo": ["lider"],
    "paciencia": ["paciencia"],
    "pensamiento creativo/crítico": ["pensamiento creativo", "pensamiento cr.tico"],
    "habilidades sociales": ["relaci", "sociales"],
    "ventas y persuasión": ["vend", "persua"],
    "visión interdisciplinaria": [".reas", "fuera"],
    "manejo de emociones": ["emociones"],
    "valentía": ["valentía"],
    "design thinking": ["design thinking"],
    "todas": ["todas", "complemento"],
    "ninguna": ["no", "ninguna", "de que habilidades hablamos"]
}

mapping_presentation_prep = {
    "ensayo y práctica": ["practic", "ensay", "expon", "grab"],
    "estudio e investigación": ["investig", "estudi", "entender"],
    "elaboración de material": ["hacer la presentaci.n", "crear la presentaci.n", "preparo mi tema", "armar un gui.n"],
    "uso de estrategias narrativas": ["storytelling", "enamoro del tema"],
    "memorización de puntos clave": ["memoriza", "prepar"],
    "manejo de la ansiedad": ["tranquiliz", "personas"],
    "no preparación": ["no "]
}

mapping_teamwork_challenges = {
    "comunicación": ["comunicaci.n", "explicar", "hablar", "efectiva"],
    "diferencias de ideas": ["diferencia de ideas", "negociar", "otras formas de hacer las cosas"],
    "falta de compromiso": ["abandono", "trabaj.. solo", "no trabajan", "apat.a", "hacer todo yo solo"],
    "organización": ["organizaci.n", "cronograma", "tareas", "calendario"],
    "colaboración con personas difíciles": ["personas complicadas", "no coopera", "no me agrada", "forzarlo"],
    "habilidades blandas": ["habilidades blandas", "adapt.ndome", "confianza", "l.mites"],
    "trato y actitud de compañeros": ["grosero", "compa.eros", "maltrato"]
}

mapping_public_speaking = {
    "miedo o ansiedad": ["nervios", "nervioso", "nerviosa", "ansioso", "intimidad.", "mal", "no me gusta"],
    "seguridad": ["preparaci.n", "m.s"],
    "inseguridad inicial pero mejora": ["al principio", "inicio", "luego agarro vuelo", "despu.s mejora"],
    "confianza": ["bien", "perfectamente", "c.modo"]
}

mapping_stress_management = {
    "sin técnicas": ["no tengo", "ninguna", "no conozco"],
    "respiración y relajación": ["respirar", "respiraci.n", "respiro", "cierro los ojos", "meditaci.n"],
    "ejercicio físico": ["ejercicio", "caminar", "salir "],
    "terapia o apoyo psicológico": ["terapia", "dtpa"],
    "autosugestión o pensamientos positivos": ["repetirme muchas veces", "recordar que eso no me va a matar"],
    "hábitos nerviosos": ["me como las u.as", "comiendo por ansiedad"],
    "descanso o pausas": ["break", "relajarme"]
}

mapping_materias = {
    "computacional": [
        "Computacional", "Computación", "Desarrollo de software", "Desarrollo de software o Ciberseguridad",
        "Seguridad Computacional", "Seguridad Computacional o bases de datos", "Bases de datos",
        "Datos y desarrollo web",  "Datos o Seguridad Computacional",
        "bases de datos", "Programación Orientada a Objetos", "POO", "Estructuras de datos", "Bases de datos",
        "Administración de bases de datos","Ingeniería de datos","Seguridad Computacional", "desarrollo web",
        "POO", "Programación",
        "Redes de computo",  "Bases de datos",
        "Metodos numéricos", "teoría de grafos", "Programación Orientada a Objetos",  "Redes de computadoras",
          "Computación", "Seguridad Computacional", "modelado matemático", "Desarrollo de software",
        "Seguridad Computacional o bases de datos", "Redes de computo", "Redes de computadoras",
        "Seguridad Computacional", "Temas selectos de computación"
    ],
    "matemáticas": [
        "matemática", "matemáticas computacionales", "matemáticas", "teoría de grafos",
        "probabilidad y estadística", "Cálculo", "Optimización", "Optimización II","matemáticas discretas",
        "Estadística", "Probabilidad", "Análisis de datos",
         "modelado matemático", "Estadística 1", "Estadística 2", "Optimización", "Cálculo",
        "Ecuaciones diferenciales","Algoritmos",
        "Álgebra lineal", "Cálculo"
    ],

    "datos": [
        "Análisis de datos", "Datos", "Ciencia de datos",
        "Ciencia de datos", "Científico de datos", "Datos", "Análisis de datos",
        "Ingeniería de datos",  "Análisis de datos", "Ciencia de datos", "Científico de datos", "Datos y desarrollo web", "Análisis de datos",
        "Ingeniería de datos", "Estadística", "Estadística y probabilidad", "modelado matemático", "Análisis de datos"
    ],

    "otras": [
         "Finanzas",  "Inglés", "Seminarios"
    ]
}

mapping_software = {
    "linux": [
        "Linux", "Linux", "Linux", "linux"
    ],
    "bases de datos": [
        "SPSS", "SQL Workbench", "MySQL Workbench", "SAS", "SQL", "SQL Workbench"
    ],
    "lenguaje de programacion": [
         "wift", "R", "c, c++", "C", "C++", "ava", "ython"
    ],
    "entorno de programacion": [
        "isual Studio", "VS Code", "isual studio", "etbeans", "Visual Studio Code",
        "olfram Mathematica", "olfram",
    ],
    "hardware": [
        "RAM", "ram", "Cisco"
    ],
    "otro": [
        "Paquete", "github", "brave", "El de optimización", "odos", "inguno",
        "Latex jaja", "No se", "Ingl.s", "Paqueter.a de Office", "aquinas", "no "
    ]
}

mapping_raiz_debido_a_materias = {
    "no debe": [
        "no debo", "No debo", "No debo", "no debo", "no", "o hay", "ninguna"
    ],
    "formas de evaluacion": [
        "valuaci.n"
    ],
    "falta de interes o dificultades": [
        "lojera", "esinterés", "enseñanza", "ificultades", "eprobado",  "culpa", "ensa",
    ],
    "pandemia": [
        "andemia", "alud"
    ],
    "problemas de tiempo": [
        "tiempo", "vida",  "poco", "Tiempo", "imult.neamente",
        " full time", "istancia"
    ],
    "problemas economicos y laborales": [
        "rabajar","rabajo", "aborales", "Laborales", "tener trabajo", "trabajo", "El trabajo", "Laborales",
        "econom.a"
    ],

    "razones personales": [
        "mpezar ", "ulpa", "nfermedad ", "mpe.o", "epresión",
        "econ.mico", "emocional", "amigo",
        "amiliares", "Sociales", "novi.s", "fiestas", "ociales", "ntrapersonales"
    ],
    "indefinido": [
        "No lo sé"
    ]
}

mapping_proyectos = {
    "preferencia varios proyectos": [
        "arios ",
        "iferente"
    ],
    "preferencia un solo proyecto": [
        "ismo",
        "olo",
    ],
    "indeterminado": [
        "Si",
        "dos",
        "ambos",
        "No se",
        "epende"
    ]
}

mapping_inspiracion_personas = {
    "familia": [
        "adres", "tíos", "abuela", "mam", "padre", "papas", "Mi mamá", "papá", "primo", "mi",
        "Mi", "mis", "Mis"
    ],
    "personas especificas": [
        "Héctor", "Bach", "maestros"
    ],
    "sin respuesta o indeterminada": [
        "nadie", "Nadie", "Aún no se", "No se"
    ],
    "variadas": [
        "varias"
    ]
}

mapping_cambios = {
    "mejoras academicas": [
        "mejores planes de estudio", "evaluación",
        "temarios", "profesores","técnicas pedagógicas",
        "enseñanza", "seríacion",
        "regularización", "clases", "diapositivas"
    ],
    "flexibilidad y tiempo": [
        "flexibilidad",  "planificar", "libre",
        "horario", "tiempo ", "separar la clase en dos horarios",
        "horarios más diversos", "línea", "laboratorio","híbrido"
    ],
    "salud mental y bienestar": [
        "menos carga", "alud mental", "descansos", "descansado",
        "auto calificables", "interactiva"
    ],
    "actitud y enfoque": [
        "ayuda", "concentrarse",
        "teórica", "preparados", "actitud", "positiva"
    ],
    "otros": [
        "problema no es el sistema", "ninguna",
         "viernes"
    ]
}

mapping_gestion_tiempo = {
    "priorizacion": [
        "prioridad",
        "Priorizar",
        "priorizo",
        "le echo ganas",
    ],
    "estructura y planificacion": [
        "cada una",
        "horario",
        "dividiendo", "organizándome"
        "limitar", "tiempo",
        "hrs"
    ],
    "equilibrio trabajo vida": [
        "equilibrar",
        "enfoco",
        "descanso",
        "me gust"
    ],
    "desorganizacion y dificultad": [
        "Aun no se gestionar",
        "Aún no tengo definido",
        "mal",
        "No lo hago",
        "como puedo"
    ],
    "tecnicas y metodos": [
        "técnica"
    ],
    "flexibilidad": [
        "epende"
    ]
}

mapping_preocupaciones = {
    "empleo": [
        "no encontrar trabajo", "quedarme sin casa", "buen empleo", "sin empleo",
        "conseguir chamba", "trabajo", "encontrar trabajo"
    ],
    "crecimiento personal": [
        "qué hacer con mi vida", "felicidad", "rumbo de la sociedad"
    ],
    "finalizacion estudios": [
        "titularme"
    ],
    "condiciones laborales": [
        "jubilarme", "no me haya gustado la carrera"
    ],
    "habilidades y preparacion": [
        "nivel de inglés"
    ],
    "sin preocupaciones": [
        "ninguna", "hasta ahorita ninguna"
    ]
}

mapping_factores_decision = {
    "economicos": [
        "dinero", "sueldo", "salario", "remuner", "subsistir", "comer"
    ],
    "familiares y ubicacion": [
        "familia", "cerca de mi casa", "ubicaci", "traslado"
    ],
    "desarrollo personal y felicidad": [
        "felicidad", "me hace realmente feliz", "gustos", "preferencias personales"
    ],
    "oportunidades y crecimiento": [
        "oportunidades", "crecimiento", "proyectos", "domin", "conocimientos"
    ],
    "sociales y emocionales": [
        "sociales", "emocionales", "interpersonales", "políticos", "naturales", "confianza"
    ],
    "trabajo actual": [
        "chamba", "trabajo actual"
    ],
    "vocacion y estudios": [
        "materias", "curs", "saber que es lo que voy a hacer el resto de mi vida"
    ]
}

mapping_situacion_profesional = {
    "camino definido": [
        "tengo un camino pensado", "ya estoy trabajando", "cuento con trabajo",
        "especialista", "claro", "empezado", "me gusta"
    ],
    "indecision o confusion": [
        "indeciso", "confundida", "no he elegido", "indefinida", "complicado elegir", "posponiendo"
    ],
    "busqueda de trabajo": [
        "buscando", "encontrar un puesto", "buenas opciones de trabajo"
    ],
    "educacion continua": [
        "maestría", "especialización", "titulo"
    ],
    "tranquilidad o sin preocupaciones": [
        "tranquila", "no me da miedo", "chida", "cómodo"
    ]
}

mapping_tipo_apoyo = {
    "transporte": [
        "transporte"
    ],
    "experiencia practica": [
        "prácticas", "rotación", "inducción profesional"
    ],
    "apoyo economico": [
        "económico", "1000000", "pesos"
    ],
    "orientacion profesional": [
        "orientación", "conferencias", "pláticas", "guía", "egresados",
        "ferias", "conocer a personas", "alguien de fuera", "hablar con los profesores"
    ],
    "capacitación": [
        "cursos", "sectores específicos", "programa para encontrar trabajos"
    ],
    "apoyo psicologico": [
        "psicologico"
    ],
    "ninguno o indeciso": [
        "ninguno", "nain", "no sé"
    ]
}

mapping_habilidades = {
    "conocimiento del campo laboral": [
        "campo laboral", "ofertas de vacantes", "futuro laboral", "sueldos",
        "perfiles de las ofertas", "actividades a realizar"
    ],
    "autoconocimiento y toma de decisiones": [
        "autoconocimiento", "saber qué quieres", "qué te gusta", "para qué eres bueno",
        "decisión acertada", "panorama completo"
    ],
    "habilidades tecnicas": [
        "programación", "estadística", "computación", "mates aplicadas"
    ],
    "habilidades blandas": [
        "pensamiento crítico", "comunicación", "dominio del inglés", "expresar lo que quiero"
    ],
    "networking": [
        "networking", "conocer personas", "red de contactos"
    ],
    "preparacion para el empleo": [
        "entrevistas", "creación de cv", "considerado en vacantes"
    ]
}

mapping_influencia = {
    "alta influencia": [
        "bastante", "influyen mucho", "demasiado", "mucho",
        "gran impacto", "mucha", "en la más alta medida"
    ],
    "media influencia": [
        "algo", "descubrir que me ayuda más", "gracias a ellas sé a dónde no quiero ir"
    ],
    "baja influencia": [
        "poco", "he hecho varios proyectos", "ninguna", "no refleja la vida laboral",
        "en ninguno", "nada"
    ],
    "inverso": [
        "el camino que elija influirá en las prácticas", "no al revés"
    ]
}

mapping_valores = {
    "salario": [
        "salario", "sueldo", "relación sueldo-carga de trabajo"
    ],
    "ambiente laboral": [
        "ambiente laboral", "flexible", "buenos horarios"
    ],
    "aprendizaje y crecimiento": [
        "seguir aprendiendo", "nuevos retos", "enseñar habilidades",
        "oportunidad de crecimiento", "posibilidad de escalar"
    ],
    "intereses personales": [
        "intereses", "hacer algo que me apasione", "valga más como persona",
        "gustos", "sumen", "proyectos"
    ],
    "seguridad empleo": [
        "empleo"
    ]
}

mapping_mejoras = {
    "prácticas profesionales": [
        "prácticas profesionales", "internships", "contacto con lo que se usa en un trabajo real",
        "acercamiento más temprano al ámbito laboral"
    ],
    "habilidades blandas": [
        "habilidades blandas", "IBM"
    ],
    "vinculación empresarial": [
        "relación laboral", "empresas", "red de contactos", "comunidad"
    ],
    "actualización plan estudios": [
        "actualización", "mejorar los temarios", "plan de estudios", "optativas",
        "temarios", "laboratorios"
    ],
    "aprendizaje práctico": [
        "más práctico", "experiencias cercanas a lo profesional", "ejemplos de la vida laboral"
    ],
    "flexibilidad horaria": [
        "oportunidades en los horarios", "clases en línea", "poder trabajar y estudiar"
    ]
}

mapping_dispositivos = {
    "telefono": ["teléfono", "móvil", "smartphone", "teléfono móvil"],
    "laptop": ["laptop", "computadora", "portátil"],
    "tablet": ["tablet"],
    "smartwatch": ["smartwatch"]
}

mapping_redes_sociales = {
    "instagram": ["instagram", "ig"],
    "facebook": ["facebook", "fb"],
    "tikTok": ["tiktok", "tik tok", "tiktok"],
    "twitter": ["twitter", "x"],
    "whatsApp": ["whatsapp"],
    "snapchat": ["snapchat"],
    "reddit": ["reddit"],
    "youTube": ["youtube"],
    "telegram": ["telegram"]
}

mapping_motivos_redes = {
    "entretenimiento": ["entretenimiento", "ocio", "divertirme", "distraído", "ver videos", "memes"],
    "comunicación": ["comunicación", "comunicarme", "mensajería", "contacto con mis amigos", "estar al tanto de mis conocidos"],
    "distracción": ["distracción", "perder el tiempo", "pos nomás"],
    "aprender": ["aprender"],
    "contenido": ["contenido"]
}

mapping_momentos_dia = {
    "mañana": ["mañana", "por la mañana"],
    "tarde": ["tarde", "por las tardes", "en la tarde"],
    "noche": ["noche", "noches", "por la noche", "en la noche"],
    "todo el día": ["todo el día", "siempre"],
    "horas libres": ["horas libres", "cuando estoy aburrido", "cuando tengo tiempo muertos", "en plazos del día"],
    "transporte": ["transporte", "ida y regreso"]
}

mapping_talleres = {
    "uso de redes y tecnología": ["uso óptimo de las redes", "tener un buen uso de estas", "uso de tecnologías", "configurar un router"],
    "ciberseguridad": ["ciberseguridad", "seguridad", "seguridad y extracción de datos"],
    "formación técnica": ["técnicos", "curso IBM"],
    "situaciones de vulnerabilidad": ["reglamento de la UNAM", "cómo reaccionar ante una situación de vulnerabilidad"],
    "ninguna": ["ninguno", "ninguna", "no lo sé", "no estaría interesada", "nada"]
}


mapping_estrategias = {
    "control del tiempo": ["medir el tiempo", "control del tiempo", "horarios establecidos",
                           "limitar para que se usa el internet", "medir el tiempo, tomar en cuenta lo utilidad de lo que se hace"],
    "educación y concientización": ["campañas", "más información", "hacer conciencia", "infografías", "talleres"],
    "acción individual y responsabilidad": ["responsabilidad de cada persona", "cada quien es responsable de lo que hace o no"],
    "uso académico del internet": ["el uso del internet esté limitado solo para lo académico", "no abusar del internet para buscar respuestas"],
    "condiciones del entorno": ["propiciar la convivencia", "no con amigos o en clases", "actividad para sustituir tiempo en tecnología"],
    "ninguna": ["ninguna", "no sé me ocurre ahora mismo", "no lo sé", "ninguno"]
}

mapping_actividades = {
    "educación y concientización": ["campañas sobre el tema", "saber el daño que podemos a hacer a otros",
                                    "seminarios", "infografías", "clases didácticas de redes responsables", "platicas", "eventos"],
    "talleres y formación Práctica": ["talleres de como implementar la tecnología", "talleres de comunicación",
                                      "talleres en la escuelas", "algún taller o actividad que nos enseñe el uso correcto", "talleres"],
    "clases en línea": ["clases en linea", "clases didácticas"],
    "sin propuesta": ["no sé me ocurre", "ninguno", "no tengo una en mente por ahora", "no tengo idea",
                      "creo que no es necesario", "no se", "ninguna"],
    "otros": ["algún evento"]
}


## Tipos de Preguntas
(para clasificar todas las respuestas de todos los cuestionarios)

In [10]:
# Una lista de diccionarios de tipos de pregunta cada cuestionario
# pregunta: tipo
question_type = [{} for questionnaire in questionnaires]
# Estos tipos pueden ser strings (trivial entenderlos) u otro diccionario
# Si son un diccionario, se trata de un mapeo para clasificación
# clasificación: [lista de palabras a buscar en la respuesta para clasificar]
# (palabras son en realidad expresiones regulares)

# 0 – CaminoProfesionalMAC1.xlsx
question_type[0] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # ¿Cuál es tu mayor preocupación al terminar tus estudios?
    mapping_preocupaciones,
    # ¿Qué factores influyen más en tu decisión sobre qué camino seguir después de egresar?
    mapping_factores_decision,
    # ¿Cuál es tu situación actual respecto a la elección de un camino profesional después de egresar?
    mapping_situacion_profesional,
    # ¿Qué tipo de apoyo crees que te ayudaría más en esta etapa de decisión?
    mapping_tipo_apoyo,
    # ¿Qué habilidades o conocimientos crees que son esenciales para tomar una decisión sobre tu especialización o carrera profesional después de egresar?
    mapping_habilidades,
    # ¿En qué medida consideras que las prácticas profesionales o proyectos extracurriculares influirán en tu decisión de que camino seguir?
    mapping_influencia,
    # ¿Cómo visualizas tu carrera profesional dentro de 5 años? ¿Qué tipo de trabajo o proyectos te gustaría estar realizando?
    mapping_materias,
    # ¿Qué es lo que más valoras en un posible empleo o área de especialización después de terminar tu carrera?
    mapping_valores,
    # ¿Qué tan útil consideras el material de estudio de la carrera para conseguir trabajo?
    mapping_intensity,
    # ¿Sientes que tu carrera universitaria te permitió desarrollar una red de contactos profesionales que te ha ayudado en tu camino laboral?
    mapping_yes_no,
    # ¿Qué aspectos de tu formación académica crees que podrían mejorarse para preparar mejor a los futuros egresados?
    mapping_mejoras,
    # ¿Consideras que el contenido de tu carrera estuvo actualizado con las necesidades actuales del mercado laboral?
    mapping_yes_no
]

# 1 – Finanzas Transparentes (Respuestas).xlsx
question_type[1] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # 1.  ¿Cuánto dinero generas al mes?
    "quantity",
    # 2. ¿Cuánto gastas mensualmente considerando todos tus gastos?
    "quantity",
    # 3. ¿Cuánto dinero logras ahorrar al mes?
    "quantity",
    # 4. ¿Tienes dinero reservado para emergencias? Si sí, ¿cuánto aproximadamente?
    "quantity",
    # 5. ¿Tu dinero tiene liquidez o está comprometido en inversiones o deudas?
    mapping_yes_no,
    # 6. ¿Cuentas con inversiones a corto plazo?
    mapping_yes_no,
    # 7. ¿Cuentas con inversiones a largo plazo?
    mapping_yes_no,
    # 8. ¿Tienes un plan personal de retiro o planeas tener uno en el futuro?
    mapping_yes_no,
    # 9. ¿Cuántas tarjetas de crédito tienes?
    "quantity",
    # 10. ¿Qué porcentaje de tu línea de crédito tienes disponible actualmente?
    "quantity",
]

# 2 – Habilidades de MAC (respuestas).xlsx
question_type[2] = [
    # Marca temporal
    "datetime",
    # Dirección de correo electrónico
    "email",
    # 1. ¿Cuál es tu numero de cuenta?
    "account_number",
    # 2. ¿Cómo describirías tu capacidad para explicar ideas complejas a personas que no son expertas en tu área?
    mapping_intensity,
    # 3. ¿Qué tan cómodo te sientes trabajando en equipo?
    mapping_intensity,
    # 4. ¿Cuál ha sido tu mayor desafío al trabajar en equipo y cómo lo superaste?
    mapping_teamwork_challenges,
    # 5.¿Sientes que la carrera de MAC te ha preparado adecuadamente para el trabajo en equipo?
    mapping_yes_no,
    # 6.  ¿Qué tan fácil o difícil te resulta establecer nuevas conexiones?
    mapping_intensity,
    # 7. ¿Cómo te sientes al tener que hablar frente a una gran audiencia?
    mapping_public_speaking,
    # 8. ¿Qué técnicas usas para manejar el estrés y mantener la calma en situaciones difíciles?
    mapping_stress_management,
    # 9.  ¿Cómo te preparas para presentaciones orales o exposiciones frente a un público amplio?
    mapping_presentation_prep,
    # 10. ¿Crees que las habilidades técnicas (programación, matemáticas, algoritmos) son suficientes para desempeñarte en el ámbito laboral sin necesidad de habilidades blandas?
    mapping_yes_no,
    # 11. ¿Has aplicado alguna de las habilidades blandas aprendidas en estos cursos en tu vida académica o personal?
    mapping_yes_no,
    # 12. ¿Cuáles consideras que son las habilidades blandas más importantes para un profesional de Matemáticas Aplicadas y Computación?
    mapping_soft_skills,
    # 13.  ¿Cuáles de las siguientes habilidades blandas crees que te faltan desarrollar más
    mapping_soft_skills,
    # 14.  ¿Qué tan preparado/a te sientes en términos de habilidades blandas para enfrentar un empleo después de graduarte?
    mapping_intensity,
    # 15.  ¿Consideras que los estudiantes de MAC suelen subestimar la importancia de las habilidades blandas?
    mapping_yes_no
]

# 3 – Impacto del Horario en el Desempeño y Bienestar de los Estudiantes de Minería de Datos (respuestas).xlsx
question_type[3] = [
    # Marca temporal
    "datetime",
    # Dirección de correo electrónico
    "email",
    # Número de cuenta
    "account_number",
    # ¿Qué tanto afecta tener una materia de 18:00 a 20:00 hrs en viernes a tu desempeño académico?
    mapping_intensity,
    # ¿Qué tanto afecta tener una materia de 18:00 a 20:00 hrs en viernes a tu desempeño personal?
    mapping_intensity,
    # ¿Qué dificultades enfrentas para asistir y concentrarte en esta materia en ese horario? (Separa cada una por comas)
    "list",
    # ¿Qué tanto impacta este horario en tu organización del tiempo para otras actividades académicas o laborales?
    mapping_intensity,
    # ¿Crees que el rendimiento en una materia como Minería de Datos se ve afectado por el horario? ¿Por qué?
    mapping_yes_no,
    # ¿Qué tanto afecta este horario a tu alimentación y descanso?
    mapping_intensity,
    # ¿Qué sugerencias darías para mejorar la experiencia de aprendizaje en esta materia considerando el horario? (Menciona las sugerencias separando por comas)
    mapping_cambios,
    # ¿Has considerado dejar esta materia debido al horario? ¿Por qué?
    mapping_yes_no,
    # ¿Qué herramientas o métodos crees que podrían hacer más llevadera la clase en este horario? (Menciona las herramientas separando por comas)
    "list",
    # ¿Cuál es tu principal medio de transporte para llegar y salir de la universidad?
    {
        "auto": ["auto", "coche"],
        "público": ["camión", "combi", "público", "metro", "FES directo"],
        "caminar": ["camin"]
    },
    # ¿Has experimentado dificultades con el transporte debido al horario de la materia? Si es así, ¿cuáles?
    mapping_yes_no,
    # ¿Cuánto tiempo tardas en llegar a casa después de la clase? (Escribe tu respuesta en minutos)
    "quantity",
    # ¿Te sientes seguro al transportarse después de las 20:00 hrs? Explica tu respuesta.
    mapping_yes_no,
    # ¿Has tenido que modificar tus rutas o modos de transporte por la hora en que termina la clase?"
    mapping_yes_no
]

# 4 – Perfil Académico (respuestas).xlsx
question_type[4] = [
    # Marca temporal
    "datetime",
    # 1. Número de Cuenta:
    "account_number",
    # 2. Edad:
    "quantity",
    # 3. Genero
    mapping_gender,
    # 4. Estado donde resides:
    {
        "ciudad de méxico": ["CDMX", "Ciudad de M.xico"],
        "estado de méxico": ["Estad. de M.xico", "Edo Mex"],
        "guerrero": ["Guerrero"]
    },
    # 5. ¿Actualmente estas laborando o realizando tu Servicio Social?
    {
        "ninguno": ["ninguno", "no", "nain"],
        "servicio social": ["servicio"],
        "trabajo": ["trabaj", "labor"],
        "ambos": ["amb.s", "dos", "si", "sí"]
    },
    # 6. ¿Cuánto tiempo en minutos dura tu recorrido a la Facultad?
    "quantity",
    # 7.  ¿Que materias de la carrera te interesaron más las de área computacional o matemática?
    mapping_materias,
    # 8. ¿Tienes Automóvil?
    mapping_yes_no,
    # 9. ¿En qué Área de especialidad de la carrera estás interesado?
    mapping_materias,
    # 10. ¿Qué materia se te dificultó más en la carrera?
    mapping_materias,
    # 11. ¿Cuales son las 3 materias que consideras que más te aportado más en tu elección de área de especialidad?
    mapping_materias,
    # 12. ¿En cuantas y cuales materias elegiste a un profesor por "barquear" la materia?
    mapping_materias,
    # 13. ¿Te consideras bueno trabajando en equipo? ¿por qué?
    mapping_yes_no,
    # 14. ¿Te consideras perfeccionista?
    mapping_yes_no,
    # 15. ¿Tus padres o alguna persona cercana se encuentra laborando en algo relacionado con el área de especialidad que elegiste?
    mapping_yes_no,
    # 16. ¿Te gustaría crecer en una empresa siempre con el mismo proyecto durante años o preferirias trabajar en varios proyectos de diferentes temas?
    mapping_proyectos,
    # 17. ¿Qué persona consideras es tu modelo a seguir?
    mapping_inspiracion_personas,
    # 18. ¿Consideras que tu equipo de computo es adecuado para el software requerido en la carrera?
    mapping_yes_no,
    # 19. ¿Qué software utilizado en la carrera consideras el mas complejo de usar?
    mapping_software,
    # 20. ¿Qué software utilizado en la carrera consideras el mas fácil de usar?
    mapping_software,
]

# 5 – Rendimiento académico (respuestas).xlsx
question_type[5] = [
    # Marca temporal
    "datetime",
    # ¿Cuál es tu promedio actual?
    "quantity",
    # Numero de cuenta
    "account_number",
    # ¿Cuántas horas dedicas al estudio semanalmente y cómo las distribuyes?
    "quantity",
    # ¿Cuántas materias debes actualmente?
    "quantity",
    # ¿Cuántas materias has reprobado durante la carrera?
    "quantity",
    # ¿Cuántos años llevas en la carrera?
    "quantity",
    # ¿Cuál es la raíz de que debas materias?
    mapping_raiz_debido_a_materias,
    # ¿Los profesores te han apoyado durante la carrera?
    mapping_yes_no,
    # ¿Qué factores externos (familiares, laborales, sociales) afectan tu desempeño académico?
    mapping_materias,
    # ¿Qué cambios o mejoras sugerirías en el sistema educativo para mejorar el rendimiento académico?
    mapping_cambios,
    # ¿Te parece que los exámenes reflejan de manera justa tu nivel de conocimiento? ¿Por qué?
    mapping_yes_no,
    # ¿Cómo gestionas tu tiempo entre tus responsabilidades académicas y otras actividades?
    mapping_gestion_tiempo,
]

# 6 – Uso de Tecnología y Redes Sociales.xlsx
question_type[6] = [
    # Marca temporal
    "datetime",
    # Número de cuenta
    "account_number",
    # ¿Cuál es tu edad?
    "quantity",
    # ¿En qué semestre te encuentras?
    "quantity",
    # ¿Qué dispositivos electrónicos utilizas con mayor frecuencia?\n(Ej. teléfono móvil, tablet, computadora, etc.)
    mapping_dispositivos,
    # ¿Cuántas horas al día usas dispositivos electrónicos para fines personales?
    "quantity",
    # ¿Utilizas dispositivos electrónicos durante las horas escolares? ¿Para qué actividades?
    mapping_yes_no,
    # ¿Qué redes sociales usas regularmente?\n(Ej: Instagram, Facebook, TikTok, Twitter, etc.)
    mapping_redes_sociales,
    # ¿Cuál es el principal motivo por el que utilizas las redes sociales?
    mapping_motivos_redes,
    # ¿En qué momentos del día sueles acceder a las redes sociales?
    mapping_momentos_dia,
    # ¿Utilizas las redes sociales mientras realizas tareas escolares o estudios? ¿Cómo influye en tu concentración?
    mapping_yes_no,
    # ¿Consideras que el uso de redes sociales afecta tu rendimiento académico? Explica tu respuesta.
    mapping_yes_no,
    # ¿Qué tipo de redes sociales prefieres y por qué?
    # ¿Has notado cambios en tu capacidad de concentración o productividad al usar redes sociales?
    mapping_yes_no,
    # ¿Crees que el uso excesivo de tecnología afecta la interacción con tus compañeros en el entorno escolar? Explica tu percepción.
    mapping_yes_no,
    # ¿Estás al tanto de las políticas o normas sobre el uso de tecnología en tu escuela? ¿Las consideras suficientes?
    mapping_yes_no,
    # ¿Recibes información o formación sobre cómo proteger tu privacidad y seguridad en línea? ¿Qué aspectos te gustaría reforzar?
    mapping_yes_no,
    # ¿Has sido testigo o víctima de ciberacoso o situaciones inseguras en redes sociales?
    mapping_yes_no,
    # ¿Te sientes preparado/a para hacer un uso responsable y seguro de la tecnología? ¿Por qué?
    mapping_yes_no,
    # ¿Qué tipo de talleres o formación adicional te gustaría recibir en relación con el uso de tecnología y redes sociales?
    mapping_talleres,
    # ¿Cómo crees que las redes sociales influyen en tu estado de ánimo o bienestar emocional?
    mapping_raiz_debido_a_materias,
    # ¿Consideras que el uso de redes sociales ha mejorado o dificultado la comunicación y relación con tus compañeros? Detalla tu experiencia
    mapping_yes_no,
    # ¿Crees que el uso excesivo de tecnología puede contribuir al aislamiento o la desconexión social? Explica tu perspectiva
    mapping_yes_no,
    # ¿Qué medidas o estrategias propondrías para promover un uso saludable de la tecnología en la escuela?
    mapping_estrategias,
    # ¿Qué actividades o programas te gustaría que se implementaran para mejorar la educación digital y el manejo de redes sociales?
    mapping_actividades
]


## Procesamiento

In [15]:
# Lista de DataFrames donde se almacenarán los datos limpios
questionnaires_clean = []
for questionnaire in questionnaires:
  # Mismo cuestionario con mismas columnas
  questionnaires_clean.append(questionnaire.iloc[0:0])

In [16]:
for questionnaire_i, questionnaire in enumerate(questionnaires):
  for question_i, question in enumerate(questionnaire):
    current_type = question_type[questionnaire_i][question_i]
    # Si el tipo es un string, no es un mapeo
    if isinstance(current_type, str):
      # falta
      continue
    # Si el tipo es un mapeo, clasificar según él
    else:
      clean_column = [classify_answer(str(answer), current_type) for answer in questionnaire.iloc[:, question_i]]
      questionnaires_clean[questionnaire_i].iloc[:, question_i] = clean_column

In [17]:
questionnaires[0]
questionnaires_clean[6]

,Marca temporal,Número de cuenta,¿Cuál es tu edad?,¿En qué semestre te encuentras?,"¿Qué dispositivos electrónicos utilizas con mayor frecuencia?\n(Ej. teléfono móvil, tablet, computadora, etc.)",¿Cuántas horas al día usas dispositivos electrónicos para fines personales?,¿Utilizas dispositivos electrónicos durante las horas escolares? ¿Para qué actividades?,"¿Qué redes sociales usas regularmente?\n(Ej: Instagram, Facebook, TikTok, Twitter, etc.)",¿Cuál es el principal motivo por el que utilizas las redes sociales?,¿En qué momentos del día sueles acceder a las redes sociales?,...,¿Estás al tanto de las políticas o normas sobre el uso de tecnología en tu escuela? ¿Las consideras suficientes?,¿Recibes información o formación sobre cómo proteger tu privacidad y seguridad en línea? ¿Qué aspectos te gustaría reforzar?,¿Has sido testigo o víctima de ciberacoso o situaciones inseguras en redes sociales?,¿Te sientes preparado/a para hacer un uso responsable y seguro de la tecnología? ¿Por qué?,¿Qué tipo de talleres o formación adicional te gustaría recibir en relación con el uso de tecnología y redes sociales?,¿Cómo crees que las redes sociales influyen en tu estado de ánimo o bienestar emocional?,¿Consideras que el uso de redes sociales ha mejorado o dificultado la comunicación y relación con tus compañeros? Detalla tu experiencia,¿Crees que el uso excesivo de tecnología puede contribuir al aislamiento o la desconexión social? Explica tu perspectiva,¿Qué medidas o estrategias propondrías para promover un uso saludable de la tecnología en la escuela?,¿Qué actividades o programas te gustaría que se implementaran para mejorar la educación digital y el manejo de redes sociales?
0,NaT,NaN,NaN,NaN,laptop,NaN,sí,instagram,entretenimiento,noche,...,no,sí,no,sí,uso de redes y tecnología,u,u,sí,educación y concientización,educación y concientización
1,NaT,NaN,NaN,NaN,laptop,NaN,u,instagram,comunicación,noche,...,no,sí,no,sí,situaciones de vulnerabilidad,no debe,no,sí,control del tiempo,u
2,NaT,NaN,NaN,NaN,telefono,NaN,sí,tikTok,distracción,u,...,no,sí,no,sí,ninguna,no debe,no,sí,control del tiempo,talleres y formación Práctica
3,NaT,NaN,NaN,NaN,u,NaN,sí,instagram,entretenimiento,noche,...,no,sí,no,sí,formación técnica,u,u,sí,educación y concientización,talleres y formación Práctica
4,NaT,NaN,NaN,NaN,telefono,NaN,u,instagram,distracción,u,...,sí,sí,no,sí,ninguna,problemas de tiempo,u,sí,ninguna,sin propuesta
5,NaT,NaN,NaN,NaN,telefono,NaN,sí,instagram,comunicación,todo el día,...,no,no,sí,sí,ninguna,u,u,sí,ninguna,sin propuesta
6,NaT,NaN,NaN,NaN,laptop,NaN,sí,instagram,entretenimiento,transporte,...,no,sí,no,sí,ninguna,no debe,no,no,uso académico del internet,sin propuesta
7,NaT,NaN,NaN,NaN,telefono,NaN,sí,instagram,distracción,u,...,no,sí,no,sí,ninguna,u,no,sí,condiciones del entorno,sin propuesta
8,NaT,NaN,NaN,NaN,telefono,NaN,no,instagram,entretenimiento,mañana,...,no,no,no,sí,u,u,u,sí,control del tiempo,educación y concientización
9,NaT,NaN,NaN,NaN,u,NaN,sí,u,entretenimiento,todo el día,...,sí,no,no,sí,u,u,no,sí,u,sin propuesta


# Auxiliar              

In [ ]:
# Generar el código de todos los cuestionarios para question_types
output = ""
for i, questionnaire in enumerate(questionnaires):
  output += f"\question_type[{i}] = [\n"
  for column in questionnaire.columns:
    output += f"    # {column}\n\n"
  else:
    output += "]\n"